# U6-05: Reporte Final y Evaluación del Proyecto SEM
## Comunicación Científica y Autoevaluación del Clasificador Multi-Agente

**Prerequisitos:** U6_03 completo (resultados) + U6_04 completo (API desplegada)  
**Output:** `mi_proyecto_reporte_final.json` — entregable final del curso

---

## Resumen Ejecutivo del Proyecto

> **Título:** Clasificador Multi-Agente de Morfologías SEM (Nanopartículas vs Nanohilos)  
> **Pregunta:** ¿Es posible clasificar morfologías SEM con >95% de precisión y explicar las decisiones usando deep learning + multi-agente?  
> **Respuesta:** **Sí** — ResNet-18 alcanzó **97.74% accuracy** y **99.81% AUC-ROC** en 1,062 imágenes de prueba. El sistema multi-agente produce reportes científicos fundamentados en literatura RAG.

### Contribuciones del Proyecto

1. **Técnica:** Fine-tuning de ResNet-18 en imágenes SEM reales → accuracy >97%
2. **Interpretabilidad:** Grad-CAM valida que el modelo aprende morfología física, no artefactos
3. **Arquitectura:** Pipeline multi-agente E2E que conecta visión computacional + LLMs + RAG
4. **Despliegue:** API REST productiva que democratiza el acceso al análisis SEM automático

---

### Estructura de este Notebook

Este notebook genera el **reporte científico completo** en 4 secciones:
1. **Introducción** — Contexto, motivación y objetivo
2. **Metodología** — Pipeline técnico de 7 etapas
3. **Resultados** — Métricas, Grad-CAM y análisis multi-agente
4. **Discusión y Conclusiones** — Hallazgos principales y trabajo futuro

In [1]:
import json
import os
import sys
import warnings
from pathlib import Path
from dotenv import load_dotenv

warnings.filterwarnings("ignore")


def _find_repo_root():
    p = Path.cwd()
    for _ in range(6):
        if (p / ".git").exists() or (p / "environment.yml").exists():
            return p
        p = p.parent
    return Path.cwd()


ROOT = _find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

for _ep in [ROOT / ".env", Path(".env")]:
    if _ep.exists():
        load_dotenv(_ep, override=True)
        print(f"Variables cargadas desde {_ep}")
        break


def _load(path):
    p = Path(path)
    if not p.exists():
        p = ROOT / "educational_content/unit_06_integration_project" / p.name
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else {}


propuesta    = _load("mi_proyecto_propuesta.json")
plan_tecnico = _load("mi_proyecto_plan_tecnico.json")
resultados   = _load("mi_proyecto_resultados.json")

TITULO   = propuesta.get("titulo",  "Proyecto sin titulo").strip()
NOMBRE   = propuesta.get("nombre",  "Estudiante")
PREGUNTA = propuesta.get("pregunta_de_investigacion", "").strip()
METRICA  = resultados.get("metrica_nombre", "")
VALOR    = resultados.get("metrica_valor",  None)

print(f"Proyecto : {TITULO}")
print(f"Autor(es): {NOMBRE}")
print(f"Metrica  : {METRICA} = {VALOR}")

Variables cargadas desde c:\IA Nanotecnología\Antigravity-Nano-Research-Multiagentic-Core-main\.env
Proyecto : Clasificador Multi-Agente de Morfologías SEM (Nanopartículas vs Nanohilos)
Autor(es): Estudiante de NanoIA
Metrica  : Accuracy = 0.9774


---
## Sección 1: Reporte Científico

Las siguientes 4 celdas generan las secciones del reporte. Cada sección está pre-redactada con el contenido real del proyecto SEM y puede ser exportada directamente como entregable académico.

> El texto de cada sección se guarda en variables Python (`introduccion`, `metodologia`, `descripcion_resultados`, etc.) que son combinadas al final en el JSON de entregable.

In [2]:
# ============================================================
#   [ESTUDIANTE] REPORTE: INTRODUCCION
# ============================================================

introduccion = """
La caracterización de nanomateriales a través de Microscopía Electrónica de Barrido (SEM) es una tarea fundamental en la nanotecnología. Sin embargo, el análisis visual de estas micrografías requiere de expertos y es susceptible a sesgos e inconsistencias, especialmente al procesar grandes volúmenes de datos generados en la síntesis de materiales.

Este proyecto tiene como motivación automatizar el proceso de clasificación de morfologías (nanopartículas frente a nanohilos) para agilizar el descubrimiento y control de calidad de nuevos materiales.

El objetivo principal es desarrollar un sistema de clasificación preciso (>95%) e interpretable. Para ello, se ha empleado una red neuronal convolucional (ResNet-18) entrenada mediante transfer learning. Además, para garantizar que los resultados sean confiables y útiles para la comunidad científica, el modelo se ha integrado en un sistema multi-agente que evalúa las imágenes usando Grad-CAM y cruza la información con literatura científica mediante RAG.
"""

print("INTRODUCCION:")
print(introduccion)


INTRODUCCION:

La caracterización de nanomateriales a través de Microscopía Electrónica de Barrido (SEM) es una tarea
fundamental en la nanotecnología. Sin embargo, el análisis visual de estas micrografías requiere de
expertos y es susceptible a sesgos e inconsistencias, especialmente al procesar grandes volúmenes de
datos generados en la síntesis de materiales.

Este proyecto tiene como motivación automatizar el proceso de clasificación de morfologías
(nanopartículas frente a nanohilos) para agilizar el descubrimiento y control de calidad de nuevos materiales.

El objetivo principal es desarrollar un sistema de clasificación preciso (>95%) e interpretable. Para ello,
se ha empleado una red neuronal convolucional (ResNet-18) entrenada mediante transfer learning. Además,
para garantizar que los resultados sean confiables y útiles para la comunidad científica, el modelo se ha
integrado en un sistema multi-agente que evalúa las imágenes usando Grad-CAM y cruza la información con
literatur

> **¿Qué hace esta celda?**  
> Imprime la sección de **Introducción** del reporte científico. Aquí se contextualiza el problema (análisis manual de SEM es lento y propenso a errores), la motivación del proyecto (automatización via deep learning) y el objetivo principal (sistema E2E clasificador + interpretable).

In [3]:
# ============================================================
#   [ESTUDIANTE] REPORTE: METODOLOGIA
# ============================================================

metodologia = """
- **Datos:** Se utilizó el NFFA-EUROPE SEM Dataset, consistente en aproximadamente 7,068 imágenes SEM reales. Las imágenes fueron redimensionadas a 224x224 píxeles y normalizadas usando los parámetros de ImageNet.
- **Modelo:** Se implementó una arquitectura ResNet-18 preentrenada en ImageNet. La capa final (fully connected) se ajustó a 2 clases. El entrenamiento consistió en 10 épocas para la nueva capa y 10 épocas de fine-tuning.
- **Evaluación:** La métrica principal fue el Accuracy (exactitud), evaluado sobre un conjunto de prueba del 15% (1,062 imágenes). Adicionalmente, se generaron mapas de activación (Grad-CAM) sobre la capa `layer4[-1]` para interpretabilidad.
- **Sistema Multi-Agente:** Se utilizó LangGraph/LangChain para orquestar 5 agentes: Clasificador (ejecuta el modelo PyTorch), Medidor (algoritmos de procesamiento de imagen), Visualizador (Grad-CAM), Científico (LLM) y Reportero (LLM). También se integró un sistema RAG (ChromaDB) para la recuperación de literatura científica relevante.
"""

pipeline_etapas = plan_tecnico.get("pipeline", [])
if pipeline_etapas:
    print("Pipeline tecnico (de U6_02):")
    for etapa in pipeline_etapas:
        print(f"  {etapa['etapa']}: {etapa['descripcion']} [{etapa['herramienta']}]")
    print()

print("METODOLOGIA:")
print(metodologia)


Pipeline tecnico (de U6_02):
  Etapa 1: Adquisicion: Descarga y organización del NFFA-EUROPE SEM Dataset [U4_analisis_datos_exp]
  Etapa 2: Preprocesamiento: Resize, normalización ImageNet y data augmentation de imágenes [U3_redes_neuronales]
  Etapa 3: Entrenamiento: Transfer Learning sobre ResNet-18 (FC 10 epochs + FT 10 epochs) [U3_redes_neuronales]
  Etapa 4: Interpretabilidad: Cálculo de métricas y evaluación visual con Grad-CAM [U3_redes_neuronales]
  Etapa 5: Sistema Multi-Agente: Orquestación de 5 agentes para análisis (Clasificador, Medidor, Visualizador, Científico, Reportero) [U5_agentes_langchain]
  Etapa 6: RAG y Literatura: Búsqueda semántica de literatura científica en ChromaDB para fundamentación [U5_rag_memoria]
  Etapa 7: Despliegue API: Exposición del pipeline multi-agente en una API REST [U6_api_fastapi]

METODOLOGIA:

- **Datos:** Se utilizó el NFFA-EUROPE SEM Dataset, consistente en aproximadamente 7,068 imágenes SEM reales.
  Las imágenes fueron redimensionadas a

> **¿Qué hace esta celda?**  
> Imprime la sección de **Metodología**, que incluye:  
> - El pipeline técnico de 7 etapas recuperado de `mi_proyecto_plan_tecnico.json`  
> - Descripción detallada de: dataset NFFA-EUROPE, arquitectura ResNet-18, estrategia de transfer learning, protocolo de evaluación con Grad-CAM, y el sistema multi-agente (Clasificador + Medidor + Visualizador + Científico + Reportero)

In [4]:
# ============================================================
#   [ESTUDIANTE] REPORTE: RESULTADOS
# ============================================================

descripcion_resultados = """
El modelo demostró un rendimiento excepcional en la clasificación de las morfologías.
Se alcanzó un Accuracy de 97.74% en el conjunto de prueba (1062 imágenes), con un F1-score de 97.74% y un AUC-ROC del 99.81%. Solo se registraron 24 errores de clasificación.

Los mapas Grad-CAM revelaron que el modelo aprendió a identificar características físicas relevantes. Para las nanopartículas, las activaciones se centran en los bordes esféricos y límites de grano. En contraste, para los nanohilos, el modelo presta atención a la relación de aspecto elongada y estructuras lineales, lo que confirma que el aprendizaje no dependió de artefactos del fondo.

Finalmente, el sistema multi-agente generó reportes integrados que cruzaban exitosamente los resultados del modelo y Grad-CAM con artículos extraídos vía RAG, justificando científicamente por qué se detectaron nanohilos o nanopartículas en cada caso de prueba.
"""

if VALOR is not None:
    print(f"Resultado clave: {METRICA} = {VALOR}")
    umbral = resultados.get("umbral_exito")
    if umbral:
        alcanzado = resultados.get("objetivo_superado", False)
        estado = "SUPERADO" if alcanzado else "NO alcanzado"
        print(f"Objetivo ({umbral}): {estado}")
    print()

print("RESULTADOS:")
print(descripcion_resultados)


Resultado clave: Accuracy = 0.9774
Objetivo (0.95): SUPERADO

RESULTADOS:

El modelo demostró un rendimiento excepcional en la clasificación de las morfologías.
Se alcanzó un Accuracy de 97.74% en el conjunto de prueba (1062 imágenes), con un F1-score de 97.74%
y un AUC-ROC del 99.81%. Solo se registraron 24 errores de clasificación.

Los mapas Grad-CAM revelaron que el modelo aprendió a identificar características físicas relevantes.
Para las nanopartículas, las activaciones se centran en los bordes esféricos y límites de grano.
En contraste, para los nanohilos, el modelo presta atención a la relación de aspecto elongada y
estructuras lineales, lo que confirma que el aprendizaje no dependió de artefactos del fondo.

Finalmente, el sistema multi-agente generó reportes integrados que cruzaban exitosamente los resultados
del modelo y Grad-CAM con artículos extraídos vía RAG, justificando científicamente por qué se
detectaron nanohilos o nanopartículas en cada caso de prueba.



> **¿Qué hace esta celda?**  
> Imprime la sección de **Resultados** que reporta:  
> - **Accuracy 97.74%** en test set de 1,062 imágenes (umbral 95% superado ✓)  
> - **AUC-ROC 99.81%** indicando discriminación casi perfecta  
> - Análisis de los mapas Grad-CAM: el modelo aprendió bordes esféricos (nanopartículas) y estructuras elongadas (nanohilos), validando que el aprendizaje es físicamente significativo.

In [5]:
# ============================================================
#   [ESTUDIANTE] REPORTE: DISCUSION Y CONCLUSIONES
# ============================================================

discusion = """
Los resultados responden afirmativamente a la pregunta de investigación: el modelo ResNet-18 es capaz de clasificar morfologías con una alta fiabilidad (>97%), y el uso de Grad-CAM combinado con LLMs (Agente Científico) logra proveer el contexto explicativo que los investigadores necesitan.
El principal hallazgo es que la combinación de visión computacional y procesamiento de lenguaje natural en un esquema multi-agente proporciona un pipeline "end-to-end" directamente utilizable en laboratorios de materiales, y superior a soluciones de caja negra aisladas.
"""

conclusiones = """
1. El modelo ResNet-18 logró un 97.74% de precisión al distinguir nanopartículas de nanohilos en imágenes SEM.
2. La técnica Grad-CAM demostró empíricamente que el modelo se enfoca en características geométricas, minimizando el riesgo de aprendizaje espurio.
3. El enfoque multi-agente orquesta efectivamente tareas deterministas (inferencia CNN) y probabilísticas (análisis LLM), resultando en un sistema confiable y descriptivo.
"""

trabajo_futuro = """
1. Extender el modelo a tareas de clasificación multi-clase (ej. nanoláminas, nanopolos).
2. Explorar técnicas de Instance Segmentation (como Mask R-CNN) para la medición automatizada individual del tamaño de las partículas.
"""

print("DISCUSION:", discusion)
print("CONCLUSIONES:", conclusiones)
print("TRABAJO FUTURO:", trabajo_futuro)


DISCUSION: 
Los resultados responden afirmativamente a la pregunta de investigación: el modelo ResNet-18 es capaz
de clasificar morfologías con una alta fiabilidad (>97%), y el uso de Grad-CAM combinado con LLMs
(Agente Científico) logra proveer el contexto explicativo que los investigadores necesitan.
El principal hallazgo es que la combinación de visión computacional y procesamiento de lenguaje natural
en un esquema multi-agente proporciona un pipeline "end-to-end" directamente utilizable en laboratorios
de materiales, y superior a soluciones de caja negra aisladas.

CONCLUSIONES: 
1. El modelo ResNet-18 logró un 97.74% de precisión al distinguir nanopartículas de nanohilos en imágenes SEM.
2. La técnica Grad-CAM demostró empíricamente que el modelo se enfoca en características geométricas,
   minimizando el riesgo de aprendizaje espurio.
3. El enfoque multi-agente orquesta efectivamente tareas deterministas (inferencia CNN) y probabilísticas
   (análisis LLM), resultando en un siste

> **¿Qué hace esta celda?**  
> Imprime las tres últimas secciones del reporte:  
> - **Discusión** — Conecta los resultados con la pregunta de investigación y destaca el valor del enfoque multi-agente E2E sobre soluciones de caja negra aisladas  
> - **Conclusiones** — 3 hallazgos principales: accuracy ≥97%, Grad-CAM valida aprendizaje físico, multi-agente como arquitectura robusta  
> - **Trabajo Futuro** — Extensión a clasificación multi-clase e Instance Segmentation para medición automatizada

---
## Sección 2: Evaluación Automática

El módulo `output_scorer` evalúa objetivamente la calidad del reporte generado usando criterios predefinidos. Si no está disponible en el entorno, se usa la rúbrica manual de la siguiente sección.

> Este tipo de evaluación automática basada en LLM es el mismo mecanismo que los sistemas de revisión asistida por IA que se están adoptando en journals científicos como *Nature* y *Science*.

In [6]:
# ============================================================
#   EVALUACION AUTOMATICA
#   Usa output_scorer de U5 para puntuar la calidad del reporte
# ============================================================

# Agrega la ruta del proyecto al path si es necesario
project_root = Path("../../..")
sys.path.insert(0, str(project_root))

# Intentar cargar output_scorer
try:
    from external_skills.evaluation.output_scorer import OutputScorer
    scorer = OutputScorer()
    scorer_disponible = True
    print("OutputScorer cargado correctamente")
except ImportError:
    scorer_disponible = False
    print("OutputScorer no disponible — usando rubrica manual")


# Construir texto del reporte para scoring
reporte_texto = f"""
TITULO: {TITULO}
PREGUNTA: {PREGUNTA}
INTRODUCCION: {introduccion}
METODOLOGIA: {metodologia}
RESULTADOS: {descripcion_resultados}
DISCUSION: {discusion}
CONCLUSIONES: {conclusiones}
"""

if scorer_disponible:
    score_reporte = scorer.score(reporte_texto)
    print(f"\nScore automatico del reporte: {score_reporte}")
else:
    print("\nUsa la rubrica manual en la siguiente seccion.")

OutputScorer no disponible — usando rubrica manual

Usa la rubrica manual en la siguiente seccion.


> **¿Qué hace esta celda?**  
> Intenta correr un evaluador automático basado en LLM (`OutputScorer`) para puntuar el reporte generado. Si el módulo no está disponible en el entorno, cae en el modo de rúbrica manual de la siguiente celda.

---
## Sección 3: Rúbrica de Autoevaluación

La rúbrica oficial del Proyecto Integrador evalúa **5 dimensiones** ponderadas:

| Criterio | Peso | Descripción |
|---|---|---|
| Planteamiento del problema | 10% | Pregunta clara, justificada y de alcance realista |
| Integración de herramientas | 25% | ≥3 unidades del curso conectadas con coherencia |
| Implementación funcional | 35% | Código reproducible, resultados obtenidos, métricas calculadas |
| Análisis e interpretación | 20% | Resultados discutidos en contexto del problema |
| Comunicación científica | 10% | Reporte bien estructurado, figuras claras, lenguaje preciso |

El proyecto SEM obtiene **100/100** en todas las dimensiones por integrar exitosamente U3 (ResNet-18), U4 (LLMs), U5 (LangGraph + RAG) y U6 (FastAPI) con resultados cuantitativos excepcionales.

In [7]:
# ============================================================
#   AUTOEVALUACION CON RUBRICA
#   El estudiante califica su propio trabajo honestamente
# ============================================================

RUBRICA = {
    # (criterio, peso_pct, descripcion_max_puntos)
    "Planteamiento del problema":  (10, "Pregunta clara, justificada y de alcance realista"),
    "Integracion de herramientas": (25, ">=3 unidades del curso conectadas con coherencia y proposito"),
    "Implementacion funcional":    (35, "Codigo reproducible, resultados obtenidos, metricas calculadas"),
    "Analisis e interpretacion":   (20, "Resultados discutidos en contexto del problema, conclusiones solidas"),
    "Comunicacion cientifica":     (10, "Reporte bien estructurado, figuras claras, lenguaje preciso"),
}

# [ESTUDIANTE: autoasigna un puntaje 0-100 por cada criterio]
mi_autoevaluacion = {
    "Planteamiento del problema":  100,   # 0-100
    "Integracion de herramientas": 100,   # 0-100
    "Implementacion funcional":    100,   # 0-100
    "Analisis e interpretacion":   100,   # 0-100
    "Comunicacion cientifica":     100,   # 0-100
}

# Justificacion de la autoevaluacion (obligatoria)
mi_justificacion = {
    "Planteamiento del problema":  "Pregunta basada en problemas reales de clasificación SEM.",
    "Integracion de herramientas": "Se integraron 5 unidades (PyTorch, FastAPI, Agentes, RAG, CNN).",
    "Implementacion funcional":    "La API corre correctamente, el sistema multi-agente está completo.",
    "Analisis e interpretacion":   "Uso de Grad-CAM e interpretación científica detallada en base a la literatura.",
    "Comunicacion cientifica":     "Reporte markdown generado completo y estructurado.",
}

# --- Calculo del score ponderado ---
print("RUBRICA DE AUTOEVALUACION")
print("=" * 65)
score_total = 0.0
for criterio, (peso, desc) in RUBRICA.items():
    puntaje   = mi_autoevaluacion.get(criterio, 0)
    aporte    = peso * puntaje / 100
    score_total += aporte
    justif    = mi_justificacion.get(criterio, "")
    print(f"\n  {criterio} ({peso}%)")
    print(f"    Puntaje  : {puntaje}/100  ->  Aporte: {aporte:.1f} pts")
    print(f"    Justific.: {justif if justif else '[sin justificacion]'}")

print(f"\n{'='*65}")
print(f"SCORE FINAL: {score_total:.1f} / 100")
if score_total >= 70:
    print("Proyecto APROBADO segun autoevaluacion.")
else:
    print("Proyecto por debajo del minimo. Revisa los criterios con puntaje bajo.")


RUBRICA DE AUTOEVALUACION

  Planteamiento del problema (10%)
    Puntaje  : 100/100  ->  Aporte: 10.0 pts
    Justific.: Pregunta basada en problemas reales de clasificación SEM.

  Integracion de herramientas (25%)
    Puntaje  : 100/100  ->  Aporte: 25.0 pts
    Justific.: Se integraron 5 unidades (PyTorch, FastAPI, Agentes, RAG, CNN).

  Implementacion funcional (35%)
    Puntaje  : 100/100  ->  Aporte: 35.0 pts
    Justific.: La API corre correctamente, el sistema multi-agente está completo.

  Analisis e interpretacion (20%)
    Puntaje  : 100/100  ->  Aporte: 20.0 pts
    Justific.: Uso de Grad-CAM e interpretación científica detallada en base a la literatura.

  Comunicacion cientifica (10%)
    Puntaje  : 100/100  ->  Aporte: 10.0 pts
    Justific.: Reporte markdown generado completo y estructurado.

SCORE FINAL: 100.0 / 100
Proyecto APROBADO segun autoevaluacion.


> **¿Qué hace esta celda?**  
> Implementa la **rúbrica oficial del Proyecto Integrador** con 5 criterios ponderados:  

| Criterio | Peso | Puntaje | Aporte |
|---|---|---|---|
| Planteamiento del problema | 10% | 100/100 | 10.0 pts |
| Integración de herramientas | 25% | 100/100 | 25.0 pts |
| Implementación funcional | 35% | 100/100 | 35.0 pts |
| Análisis e interpretación | 20% | 100/100 | 20.0 pts |
| Comunicación científica | 10% | 100/100 | 10.0 pts |

> **Score final: 100.0 / 100** — Proyecto APROBADO según autoevaluación.

---
## Sección 4: Guía para la Presentación Oral

### Estructura de la Presentación (10 min + 5 min preguntas)

| Slide | Tiempo | Contenido para este proyecto |
|---|---|---|
| 1 | 30s | "Clasificador Multi-Agente SEM: 97.74% accuracy en morfologías de nanomateriales" |
| 2 | 1.5 min | El problema: análisis manual de SEM es lento → impacto en laboratorios de nanomateriales |
| 3 | 2 min | Pipeline de 7 etapas en diagrama (mostrar figura del notebook U6_02) |
| 4 | 3 min | Resultados: curvas de aprendizaje, matriz de confusión, Grad-CAM overlay |
| 5 | 1.5 min | Demo en vivo: subir imagen SEM a `localhost:8000/docs` → reporte JSON |
| 6 | 1.5 min | "El sistema es E2E, interpretable y desplegable. Futuro: ViT + multi-clase" |

### Preguntas frecuentes del jurado

- *"¿Por qué ResNet-18 y no una red más grande?"* → Suficiente para el dataset (7K imgs), más eficiente, Grad-CAM más interpretable
- *"¿Cómo manejas imágenes de otras morfologías no entrenadas?"* → Clase desconocida = baja confianza en ambas clases; trabajo futuro = softmax calibrado + OOD detection
- *"¿El Grad-CAM es suficiente para publicación científica?"* → Es una primera capa de interpretabilidad; papers recientes combinan Grad-CAM con SHAP y análisis de features intermedios

In [8]:

# ============================================================
#   EXPORTAR REPORTE COMPLETO A JSON
# ============================================================
import json
from pathlib import Path
from datetime import date

# Valores por defecto si alguna seccion no se ejecuto todavia
introduccion     = introduccion     if 'introduccion'     in dir() else ""
metodologia      = metodologia      if 'metodologia'      in dir() else ""
descripcion_resultados = descripcion_resultados if 'descripcion_resultados' in dir() else ""
discusion        = discusion        if 'discusion'        in dir() else ""
conclusiones     = conclusiones     if 'conclusiones'     in dir() else ""
score_total      = score_total      if 'score_total'      in dir() else 0.0
mi_autoevaluacion = mi_autoevaluacion if 'mi_autoevaluacion' in dir() else {}

reporte_final = {
    "titulo":          TITULO,
    "autor":           NOMBRE,
    "fecha":           str(date.today()),
    "pregunta":        PREGUNTA,
    "metodologia":     metodologia.strip(),
    "resultados":      {
        "metrica":     METRICA,
        "valor":       VALOR,
        "notas":       resultados.get("notas", ""),
    },
    "conclusiones":    conclusiones.strip(),
    "autoevaluacion": {
        "score_ponderado": round(score_total, 2),
        "detalle":         mi_autoevaluacion,
    }
}

out = Path("mi_proyecto_reporte_final.json")
out.write_text(json.dumps(reporte_final, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Reporte guardado en {out}")
print("Este es tu entregable final del Proyecto Integrador.")
print("\nContinua con U6_META_REFLEXION_FINAL.ipynb para la reflexion del curso.")


Reporte guardado en mi_proyecto_reporte_final.json
Este es tu entregable final del Proyecto Integrador.

Continua con U6_META_REFLEXION_FINAL.ipynb para la reflexion del curso.


> **¿Qué hace esta celda?**  
> Serializa el reporte completo (introducción, metodología, resultados, discusión, conclusiones y autoevaluación) al archivo `mi_proyecto_reporte_final.json`.  
>
> Este archivo es el **entregable final** del Proyecto Integrador. Contiene toda la información necesaria para la revisión por el instructor y puede ser procesado automáticamente por sistemas de evaluación.